# Vestibular modeling

We model vestibular system responses from papers by Ormsby (Model of human dynamic orientation, 1974)and Young and Oman (Model for vestibular adaptation to
horizontal rotation, 1970).
Specifically, the specific force actual ($f_g = a - g$) and the measured specific force $\widehat{f}_g$ are related by the transfer function
$$
  \frac{\widehat{f}_g}{f_g} = \frac{0.911 \, (s + 0.0988)}{(s + 0.133) \, (s + 1.95)}.
$$
The perceived angular velocity $\widehat{\omega}$ is related to the actual angular velocity $\omega$ by
$$
  \frac{\widehat{\omega}}{\omega} = \frac{10.3 \, s}{(10.2 \, s + 1) \, (0.1 \, s + 1)} \, \frac{30 \, s}{30 \, s + 1}.
$$
These give the components of the perceived vestibular responses, e.g., $\omega \in \{\omega_x, \omega_y, \omega_z\}$.

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate as sci_int
import scipy.linalg as sci_lin
import jax.numpy as jnp

import exp_mpc.stewart_min.comp as comp

In [ ]:
s = ct.tf("s")
assert isinstance(s, ct.TransferFunction)

acc_transfer = (0.911 * (s + 0.0988)) / ((s + 0.133) * (s + 1.95))
omega_transfer = (10.3 * s * 30 * s) / ((10.2 * s + 1) * (0.1 * s + 1) * (30 * s + 1))
assert isinstance(acc_transfer, ct.TransferFunction)
assert isinstance(omega_transfer, ct.TransferFunction)

acc_ss = acc_transfer.to_ss()
omega_ss = omega_transfer.to_ss()
assert isinstance(acc_ss, ct.StateSpace)
assert isinstance(omega_ss, ct.StateSpace)

## Brute integration scheme

We throw the linear dynamical system into a general-purpose integration scheme, `quad_vec`.

In [ ]:
res = sci_int.solve_ivp(
    fun=lambda t, x: omega_ss.A @ x + np.ravel(omega_ss.B) * (0.0 - t) * (t - 8.0) / 100.0,
    t_span=[0.0, 10.0],
    y0=np.zeros(3),
    dense_output=True,
)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=[7, 4])
ts = np.linspace(0.0, 10.0, num=2**12)
ax.plot(ts, (0.0 - ts) * (ts - 8.0) / 100.0, label="real")
ax.plot(ts, np.transpose(omega_ss.C @ res.sol(ts)), label="model")
ax.grid()
ax.legend()
plt.show()

## Fast integration scheme

Compute the matrix exponential for a small-time step $\Delta t$, and then iteratively solve the corresponding initial value problem.
We explicitly spell this out.
Given an LTI system
$$
  \dot{x} = A x + B u, \quad x(0) = x_0,
$$
the solution is
$$
  x(t) = e^{A t} x_0 + \int_0^t e^{A \, (t - \tau)} \, B u(\tau) \operatorname{d}\!\tau.
$$
Let $0 = t_0 < t_1 < \ldots < t_N = T$ be a partition with each difference $t_k - t_{k - 1} = \Delta t$ constant.
Suppose that $u(t) \equiv u_k$ is a constant on $[t_{k - 1}, t_k]$.
Define $x_k := x(t_k)$, where $x_0$ is given.
Define the (constant) matrices
$$
  E_0 = e^{A \, \Delta t} \quad\text{and}\quad E_1 = \int_0^{\Delta t} e^{A \, (\Delta t - \tau)} B \operatorname{d}\!\tau.
$$
(Note that multiplication of $B$ in $E_1$.)
Then
$$
  x_k = E_0 x_{k - 1} + E_1 u_k, \quad k = 1, \ldots, n.
$$

In [ ]:
def fast_int(
    E0: np.ndarray,
    E1: np.ndarray,
    x0: np.ndarray,
    u: np.ndarray
) -> np.ndarray:
    x0 = np.ravel(x0)
    u = np.ravel(u)
    E1 = np.ravel(E1)

    assert E0.shape[0] == E0.shape[1]
    assert E1.size == E0.shape[1]
    assert x0.size == E0.shape[1]
    assert u.size > 0

    x = np.empty(shape=(u.size + 1, x0.size), dtype=float)
    x[0] = x0
    for i in range(u.size):
        x[i + 1] = E0 @ x[i] + E1 * u[i]
    return x

@jax.jit
def fast_int_jax(
    E0: jax.Array,
    E1: jax.Array,
    x0: jax.Array,
    u: jax.Array,
) -> jax.Array:
    x0 = jnp.ravel(x0)
    u = jnp.ravel(u)
    E1 = jnp.ravel(E1)

    assert E0.shape[0] == E0.shape[1]
    assert E1.size == E0.shape[1]
    assert x0.size == E0.shape[1]
    assert u.size > 0

    x = jnp.empty(shape=(u.size + 1, x0.size), dtype=float)
    x = x.at[0].set(x0)

    def for_body(i: int, x: jax.Array) -> jax.Array:
        xi = E0 @ x[i] + E1 * u[i]
        x = x.at[i + 1].set(xi)
        return x

    x = jax.lax.fori_loop(0, u.size, for_body, x)
    return x

@jax.jit
def fast_int_jax2(
    E0: jax.Array,
    E1: jax.Array,
    C: jax.Array,
    D: jax.Array,
    x0: jax.Array,
    u: jax.Array,
) -> jax.Array:
    x0 = jnp.ravel(x0)
    u = jnp.ravel(u)
    E1 = jnp.squeeze(E1)
    C = jnp.squeeze(C)

    assert E0.shape[0] == E0.shape[1]
    assert E1.size == E0.shape[1]
    assert C.size == E0.shape[1]
    assert x0.size == E0.shape[1]
    assert u.size > 0

    x = jnp.empty(shape=(u.size + 1, x0.size), dtype=float)
    x = x.at[0].set(x0)

    def for_body(i: int, x: jax.Array) -> jax.Array:
        xi = E0 @ x[i] + E1 * u[i]
        x = x.at[i + 1].set(xi)
        return x

    # skip intial condition, because can't optimize over it, and it isn't
    #  really meaninful in the context of having a matrix D
    x = jax.lax.fori_loop(0, u.size, for_body, x)
    x = x[1:]
    y = C @ x.T + np.squeeze(D @ jnp.atleast_2d(u))
    return y

In [ ]:
def get_E0_E1(ss: ct.StateSpace, dt: float) -> tuple[np.ndarray, np.ndarray]:
    E0 = sci_lin.expm(ss.A * dt)
    E1 = sci_int.quad_vec(
        f=lambda t: sci_lin.expm(ss.A * (dt - t)) @ ss.B,
        a=0,
        b=dt,
    )[0]
    return E0, E1

In [ ]:
# omega case
N = 2**10 - 1
T = 200.0
dt = T / N
E0, E1 = get_E0_E1(omega_ss, dt)

In [ ]:
fast_x = fast_int_jax(E0, E1, np.zeros(3), 10.0 * np.ones(N))
fast_y = fast_int_jax2(E0, E1, omega_ss.C, omega_ss.D, np.zeros(3), 10.0 * np.ones(N))
fast_y

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ts = np.linspace(0.0, T, N + 1)
ax.plot(ts, fast_y)
ax.grid()
plt.show()

## Generate matrices for MPC

In [ ]:
import exp_mpc.stewart_min.const as const

E0_omega, E1_omega = get_E0_E1(omega_ss, const.dt)
E0_acc, E1_acc = get_E0_E1(acc_ss, const.dt)

In [ ]:
E0_omega, E1_omega, omega_ss.C

In [ ]:
E0_acc, E1_acc, acc_ss.C

## Faster integration scheme

We introduce fancy linear-algebraic update scheme.
Consider the recursive problem
$$
  x_k = E_0 x_{k - 1} + E_1 u_{k - 1},
$$
with given $u_k$ and $x_0$.
We want to update this efficiently, both in the forward pass and in the backpropagation of gradients.
The following scheme is posited.
Suppose that $E_0$ is diagonalizable, say
$$
  E_0 = P D P^{-1},
$$
with $D$ diagonal and with $P$ the corresponding eigenvectors.
If we introduce the notation $\tilde{x}_k = P^{-1} x_k$ and $\tilde{u}_k = P^{-1} E_1 u_k$, then we have the update rules
$$
  \tilde{x}_k = D \tilde{x}_{k - 1} + \tilde{u}_{k - 1}, \quad \tilde{x}_0 = P^{-1} x_0.
$$
again, with $D$ diagonal.
So, these update rules can be applied componentwise.
Simply counting floating point operations shows that this is more efficient (by a constant factor).
More importantly, the back propagation of gradients rule is very simple, because we are simply acting componentwise in our updates, most of the time.
To get our desired observed variables, we have the conversion
$$
  y_k = C P \tilde{x}_k.
$$
Numerical experiments (done in a temporary notebook) show that this update scheme is very stable (for typical horizon lengths).
Numerical experiments also show that this scheme backpropagates gradients about 4 times faster.

In [ ]:
tmp = sci_lin.eig(const.E0_acc)
D_acc, P_acc = tmp[0], tmp[1]
P_acc_inv = np.linalg.inv(P_acc)
D_acc = D_acc.real  # type: ignore
assert np.max(np.abs(P_acc @ np.diag(D_acc.real) @ P_acc_inv - const.E0_acc)) < 1e-15

tmp = sci_lin.eig(const.E0_omega)
D_omega, P_omega = tmp[0], tmp[1]
P_omega_inv = np.linalg.inv(P_omega)
D_omega = D_omega.real  # type: ignore
assert np.max(np.abs(P_omega @ np.diag(D_omega.real) @ P_omega_inv - const.E0_omega)) < 1e-15

EP1_acc = np.linalg.inv(P_acc) @ const.E1_acc
CP_acc = const.C_acc @ P_acc
EP1_omega = np.linalg.inv(P_omega) @ const.E1_omega
CP_omega = const.C_omega @ P_omega

In [ ]:
D_acc, P_acc_inv, EP1_acc, CP_acc

In [ ]:
D_omega, P_omega_inv, EP1_omega, CP_omega

## Observability

Given a discrete system
\begin{align*}
  x_{k + 1} &= A x_k + B u_{k + 1} \\
  y_{k + 1} &= C x_{k + 1},
\end{align*}
the hidden states $x_k$ are unknown, whereas the $y_k$ and $u_k$ are known.
In the angular velocity case, we have $x_k \in \mathbb{R}^3$, $u_k \in \mathbb{R}$, and $y_k \in \mathbb{R}$.
Simply applying the recurrence relation several times yields
$$
  \begin{bmatrix} y_1 \\ y_2 \\ y_3 \end{bmatrix} = \begin{bmatrix} C \\ C A \\ C A^2 \end{bmatrix} x_1 + \begin{bmatrix} 0 & 0 \\ C B & 0 \\ C A B & C B \end{bmatrix} \begin{bmatrix} u_2 \\ u_3 \end{bmatrix}.
$$
So, if the observability matrix
$$
  \mathcal{O} := \begin{bmatrix} C \\ C A \\ C A^2 \end{bmatrix}
$$
is invertible, the initial state becomes computable:
$$
  x_1 = \begin{bmatrix} C \\ C A \\ C A^2 \end{bmatrix}^{-1} \bigg(\begin{bmatrix} y_1 \\ y_2 \\ y_3 \end{bmatrix} - \begin{bmatrix} 0 & 0 \\ C B & 0 \\ C A B & C B \end{bmatrix} \begin{bmatrix} u_2 \\ u_3 \end{bmatrix}\bigg).
$$
The other hidden variables can be subsequently computed via the defining recurrence relation.
E.g., for $x_2$ and $x_3$, compute
$$
  x_2 = A x_1 + B u_2 \quad\text{and}\quad x_3 = A x_2 + B u_3.
$$
We verify the accuracy of this scheme for the fast LTI integration case above.
In this case, $A = E_0$, $B = E_1$, and $C = C$.

In [ ]:
@jax.jit
def obs_x1(
    E0: jax.Array,
    E1: jax.Array,
    C: jax.Array,
    y: jax.Array,
    u: jax.Array,
) -> jax.Array:
    """Returns initial hidden state x1 corresponding to y1."""
    n = E0.shape[0]
    mpow = jnp.linalg.matrix_power
    squee = jnp.squeeze

    y = jnp.ravel(y)
    u = jnp.ravel(u)
    assert y.size == n
    assert u.size == n - 1

    # For `n == 3`, we have the following matrices:
    # `
    # O = jnp.vstack([C, C @ E0, C @ E0 @ E0])
    # U = jnp.array([
    #     [0., 0.],
    #     [jnp.squeeze(C @ E1), 0.],
    #     [jnp.squeeze(C @ E0 @ E1), jnp.squeeze(C @ E1)],
    # ])
    # `
    # The following code is valid for general `n`.

    O = jnp.vstack([C @ mpow(E0, i) for i in range(n)])  # noqa: E741
    U_vals = [0., 0.] + [squee(C @ mpow(E0, i) @ E1) for i in range(n - 1)]
    U = jnp.array([
        [U_vals[j] for j in range(i + 1, i + 1 - (n - 1), -1)] for i in range(n)
    ])
    return jnp.linalg.solve(O, y - U @ u)

In [ ]:
u = jnp.array(10.0)
obs_x = np.zeros_like(fast_x)
for i in range(fast_y.size - 5):
    # technically, the last couple of values for obs_x are meaningless
    obs_x[i] = obs_x1(E0, E1, omega_ss.C, fast_y[i: i + 3], jnp.array([u, u]))
obs_x = jnp.array(obs_x)
obs_y = omega_ss.C @ obs_x.T

In [ ]:
print(f"err_y = {float(jnp.max(jnp.abs(jnp.ravel(obs_y - fast_y)[:-5])))}\n"
      f"err_x = {float(jnp.max(jnp.abs(jnp.ravel((obs_x - fast_x)[:-5]))))}")

## initial starting conditions

Recall the transfer function
$$
  f(s) := \frac{\widehat{f_g}}{f_g} = \frac{0.911 \, (s + 0.0988)}{(s + 0.133) \, (s + 1.95)}.
$$
For a constant signal $f_g \equiv \mathrm{const}$, we have $\widehat{f}_g(t) \to f_g \, f(0)$, where
$f(0) \doteq 0.347047619047619$.
We use this to find the internal states at this steady state.

In [ ]:
earth_gravity = const.gravity[2]
moon_gravity = np.array(1.625)  # m / s

fac = (0.911 * 0.0988) / (0.133 * 1.95)

v0_earth = obs_x1(const.E0_acc, const.E1_acc, const.C_acc, np.ones(2) * fac * earth_gravity, np.ones(1) * earth_gravity)
v0_moon = obs_x1(const.E0_acc, const.E1_acc, const.C_acc, np.ones(2) * fac * moon_gravity, np.ones(1) * moon_gravity)
v0_earth, v0_moon

In [ ]:
err0 = comp.lti_int_single(const.E0_acc, const.E1_acc, v0_earth, earth_gravity) - v0_earth
err1 = comp.lti_int_single(const.E0_acc, const.E1_acc, v0_moon, moon_gravity) - v0_moon
np.max(np.abs(np.concatenate([err0, err1])))